# Calculate regional mortality with parametric bootstrapping

This may require large memory. Can take up to 20-24hrs for one health outcome/ensemble member.

In [ ]:
import os
import xarray as xr
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import mortality

In [ ]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"
TMREL_DIR = "/glade/derecho/scratch/awells/air_quality/TMREL/"
RR_DIR = "/glade/derecho/scratch/awells/air_quality/rr_pm25/"

In [ ]:
# Load country masks
mask_file = "GBD_Region_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [ ]:
# === Calculate the scalar distributions ===
n_samples = 200

# TMREL from GBD21 (uniform distribution)
tmrel_file = f"TMREL_{n_samples}_samples_pm25.nc"
tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
tmrel_da = xr.open_dataarray(tmrel_path)

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop}"

PM25_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25_bc/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/region/{n_samples}_samples"

### Health variables
"COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE", "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"

It is best to run for just one ensemble member and mortality outcome at a time

In [ ]:
health_VAR = "COPD"
ens_num = [1]

In [ ]:
print(f"Processing health variable {health_VAR}")

# RR from GBD21 (normal distribution)
rr_file = f"RR_{health_VAR}_{n_samples}_samples_pm25.nc"
rr_path = os.path.join(RR_DIR, rr_file)
rr_da = xr.open_dataarray(rr_path)

del rr_file, rr_path

# Load BMR for each grid point (normal distribution)
bmr_file = f"GBD_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

del bmr_file, bmr_path

print(f"Processing ensemble member {ens_num:02d}")

# Load ozone data
pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
pm25_path = os.path.join(PM25_DIR, pm25_file)
pm25 = xr.open_dataarray(pm25_path).astype("float32")

# Adjust indices to match (with small tolerance)
# e.g., max 1e-9 km distance
pm25 = pm25.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

del pm25_file, pm25_path

# make pm25 dask-backed
pm25 = pm25.chunk({'lat': 180, 'lon': 360})

for year in range(years.start, years.stop + 1):
    print(f"Processing year {year}")

    # Find RR at each grid point
    pm25_year = pm25.sel(year=year)
    RR = rr_da.interp(exposure=pm25_year)

    del pm25_year

    # Scale the RR based on TMREL, i.e. RR=1 when exposure <= TMREL
    RR = RR.where(RR["exposure"] >= tmrel_da, 1)
    # Calculate the attributable fraction
    AF = (1 - (1/RR)).chunk({"samples": 10})

    del RR

    POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})

    # Calculate mortality at each grid point for n samples
    M = mortality(AF, BMR, POP)

    del AF, POP

    # Calculate regional mortality - save per year per region
    for region in masks.region:
        print(f"Processing {region.data}")

        region_mask = masks.sel(region=region)
        region_M = M.where(region_mask == 1).sum(dim=("lat", "lon"))
        region_M = region_M.expand_dims(region=[region.data])

        del region_mask

        description = (f"Regional {health_VAR} mortality due to PM2.5 "
                       f"sample size {n_samples} - scripts by A.F."
                       " Wells (2025)")
        region_M.attrs["description"] = description
        region_M.attrs["model"] = model
        region_M.attrs["scenario"] = scenario
        region_M.attrs["ensemble_number"] = ens_num
        region_M.attrs["region"] = region.data
        region_M.attrs["year"] = year

        out_file = f"Regional_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{region.data}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving to {out_path}")
        region_M.to_netcdf(out_path)

        del region_M

del pm25

del rr_da, BMR

print("All processing complete.")